# Gold: patient_360

Builds `gold.patient_360` (one row per patient) from silver `patients`, `hospitals`, `insurance_providers` and aggregated `lab_results`, per the STTM `sttm_silver_to_gold.xlsx`. Load strategy: full rebuild (CREATE OR REPLACE TABLE ... AS SELECT); all joins are LEFT JOINs from `patients`.

Note: silver tables currently contain duplicate rows per business key, so each source is deduplicated to its latest record before joining to preserve the grain.

In [0]:
import re

# <catalog>/silver/gold placeholders from the STTM, mapped to this workspace's Unity Catalog names
dbutils.widgets.text("catalog", "man_catlog")
dbutils.widgets.text("silver_schema", "caresync_schema_silver")
dbutils.widgets.text("gold_schema", "caresync_schema_gold")

def _ident(name):
    v = dbutils.widgets.get(name).strip()
    if not re.fullmatch(r"[A-Za-z0-9_]+", v):
        raise ValueError(f"Invalid identifier for {name}: {v!r}")
    return v

catalog = _ident("catalog")
silver = f"{catalog}.{_ident('silver_schema')}"
gold = f"{catalog}.{_ident('gold_schema')}"
target_table = f"{gold}.patient_360"
print(f"Source: {silver} | Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per patient with hospital, insurance and lab-test summary (STTM silver->gold v1.0)'
AS
WITH patients AS (
  -- Silver holds duplicate rows per key; keep the latest record per patient
  SELECT * FROM {silver}.patients
  QUALIFY ROW_NUMBER() OVER (PARTITION BY patient_id ORDER BY updated_at DESC, landing_ts DESC) = 1
),
hospitals AS (
  SELECT * FROM {silver}.hospitals
  QUALIFY ROW_NUMBER() OVER (PARTITION BY hospital_id ORDER BY updated_at DESC, landing_ts DESC) = 1
),
insurance_providers AS (
  SELECT * FROM {silver}.insurance_providers
  QUALIFY ROW_NUMBER() OVER (PARTITION BY insurance_provider_id ORDER BY landing_ts DESC) = 1
),
lab_results AS (
  SELECT * FROM {silver}.lab_results
  QUALIFY ROW_NUMBER() OVER (PARTITION BY lab_result_id ORDER BY reported_at DESC, landing_ts DESC) = 1
),
lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count,
    MAX(test_date)                        AS last_test_date
  FROM lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                                    AS patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                                AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  CAST(p.registered_hospital_id AS INT)                                        AS hospital_id,
  h.hospital_name,
  CAST(p.insurance_provider_id AS INT)                                         AS insurance_provider_id,
  i.provider_name                                                              AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                      AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                              AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                              AS critical_test_count,
  CAST(l.last_test_date AS DATE)                                               AS last_test_date
FROM patients p
LEFT JOIN hospitals h           ON p.registered_hospital_id = h.hospital_id
LEFT JOIN insurance_providers i ON p.insurance_provider_id  = i.insurance_provider_id
LEFT JOIN lab_agg l             ON p.patient_id             = l.patient_id
""")
print(f"Created {target_table}")

In [0]:
chk = spark.sql(f"""
SELECT
  (SELECT COUNT(*) FROM {target_table})                                   AS gold_rows,
  (SELECT COUNT(DISTINCT patient_id) FROM {target_table})                 AS gold_distinct_patients,
  (SELECT COUNT(DISTINCT patient_id) FROM {silver}.patients)              AS silver_distinct_patients,
  (SELECT SUM(total_tests) FROM {target_table})                           AS gold_total_tests,
  (SELECT COUNT(DISTINCT lab_result_id) FROM {silver}.lab_results
     WHERE patient_id IN (SELECT patient_id FROM {silver}.patients))      AS silver_tests_for_known_patients
""").first()
display(spark.createDataFrame([chk]))
assert chk.gold_rows == chk.gold_distinct_patients == chk.silver_distinct_patients, "Grain violated: expected one row per patient"
assert chk.gold_total_tests == chk.silver_tests_for_known_patients, "Lab test totals do not reconcile with silver"
print("Validation passed")

In [0]:
display(spark.table(target_table).orderBy("patient_id").limit(20))